# Fine-Tuning Kong's Piano Model for Guitar Audio-to-MIDI

Clean implementation following TART (arXiv:2510.02597) + Riley et al. (ICASSP 2024).

**Proven result:** fine-tuning Kong's model on GuitarSet+EGDB lifted note F50 from 0.704 → 0.838 on GuitarSet.

**Pipeline:** Mount Drive → copy to local SSD → build dataset → load Kong model → fine-tune → evaluate P50/R50/F50 vs baseline.

## 1. Install dependencies

In [2]:
!pip uninstall torch torchaudio -y
!pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu124

Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 114.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 65.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 152.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 47.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 21.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━

In [1]:
!nvidia-smi

Sun Jun 21 18:41:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   40C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
!pip -q install piano_transcription_inference librosa pretty_midi mir_eval soundfile
!pip -q install torch torchaudio --index-url https://download.pytorch.org/whl/cu128

import torch
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 131.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 6.4 MB/s eta 0:00:00
PyTorch: 2.11.0+cu128 | CUDA: True
Device: cuda


## 2. Mount Drive + copy data to local SSD

In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted.')
except ModuleNotFoundError:
    print('Not in Colab.')

import os, glob, shutil, json
from pathlib import Path

# Locate GuitarSet on Drive
DATA_ROOT_CANDIDATES = [
    Path('/content/drive/MyDrive/Capstone/FullGuitarSetData'),
    Path('/content/drive/MyDrive/FullGuitarSetData'),
    Path('/content/drive/MyDrive/Capstone/GuitarSet'),
]
DATA_ROOT = next((c for c in DATA_ROOT_CANDIDATES if (c/'JamsFiles').exists()), None)
if DATA_ROOT is None: raise FileNotFoundError("GuitarSet not found on Drive")
print(f"Found GuitarSet at: {DATA_ROOT}")

# Copy to local SSD (fast local reads during training)
LOCAL_AUDIO = '/content/gs_audio'
LOCAL_JAMS  = '/content/gs_jams'
os.makedirs(LOCAL_AUDIO, exist_ok=True)
os.makedirs(LOCAL_JAMS,  exist_ok=True)

def copy_if_needed(src_dir, dst_dir, ext):
    src_files = glob.glob(os.path.join(str(src_dir), f'*.{ext}'))
    dst_files = glob.glob(os.path.join(dst_dir, f'*.{ext}'))
    if len(dst_files) >= len(src_files):
        print(f"  {ext}: already on local SSD ({len(dst_files)} files)")
        return
    print(f"  Copying {len(src_files)} {ext} files to local SSD...")
    for f in src_files:
        try: shutil.copy2(f, dst_dir)
        except shutil.SameFileError: pass
    print(f"  Done.")

copy_if_needed(DATA_ROOT/'AudioFiles', LOCAL_AUDIO, 'wav')
copy_if_needed(DATA_ROOT/'JamsFiles',  LOCAL_JAMS,  'jams')
print(f"Audio: {len(glob.glob(LOCAL_AUDIO+'/*.wav'))} | JAMS: {len(glob.glob(LOCAL_JAMS+'/*.jams'))}")

OUTPUT_DIR = '/content/drive/MyDrive/Capstone/outputs/kong_finetune'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Checkpoints -> {OUTPUT_DIR}")

Mounted at /content/drive
Drive mounted.
Found GuitarSet at: /content/drive/MyDrive/Capstone/FullGuitarSetData
  Copying 360 wav files to local SSD...
  Done.
  Copying 360 jams files to local SSD...
  Done.
Audio: 360 | JAMS: 360
Checkpoints -> /content/drive/MyDrive/Capstone/outputs/kong_finetune


## 3. Config — TART's proven hyperparameters

In [19]:
# TART recipe (arXiv:2510.02597 Table 2):
# 10s segments / 1s hop / +-2 semitone pitch shift / batch 4
# lr 1e-5 * 0.9 every 10k steps / 100k steps / NO onset jitter
# Acoustic-Electric (GuitarSet+EGDB combined) = best result

SR          = 16000   # Kong model sample rate (differs from Basic Pitch's 22050)
SEGMENT_SEC = 10.0
SEGMENT_SAMPLES = int(SEGMENT_SEC * SR)  # 160000
N_FRAMES = 1000  # fixed model output size (ignore the +1 from earlier)HOP_SEC     = 1.0
SEGMENT_SAMPLES = int(SEGMENT_SEC * SR)
MIDI_MIN    = 21      # A0
MIDI_MAX    = 108     # C8
N_NOTES     = 88
ONSET_TOL   = 0.05    # +/-50ms for P50/R50/F50
#N_FRAMES = 501

PITCH_SHIFTS  = (-2, -1, 0, 1, 2)   # semitones; onset jitter OMITTED (TART: harmful)
BATCH_SIZE    = 1
LR            = 1e-5
LR_DECAY      = 0.9
DECAY_STEPS   = 10_000
TOTAL_STEPS   = 100_000
VAL_EVERY     = 2_000

JAMS_DIR  = LOCAL_JAMS
AUDIO_DIR = LOCAL_AUDIO
print("Config ready.")
print(f"SR={SR}Hz | segment={SEGMENT_SEC}s | batch={BATCH_SIZE} | lr={LR}")

Config ready.
SR=16000Hz | segment=10.0s | batch=1 | lr=1e-05


## 4. Data loading — GuitarSet JAMS parser

In [4]:
def load_guitarset_notes(jams_path):
    """Read GuitarSet JAMS as raw JSON; merge note_midi across all 6 strings."""
    with open(jams_path) as f:
        jam = json.load(f)
    notes = []
    for ann in jam.get('annotations', []):
        if ann.get('namespace','') not in ('note_midi','pitch_midi'): continue
        for obs in ann['data']:
            notes.append({'onset':  float(obs['time']),
                          'offset': float(obs['time']) + float(obs['duration']),
                          'midi':   int(round(float(obs['value'])))})
    return sorted(notes, key=lambda n: (n['onset'], n['midi']))

def build_index(audio_dir, jams_dir):
    records = []
    for jp in sorted(glob.glob(os.path.join(jams_dir, '*.jams'))):
        stem = os.path.splitext(os.path.basename(jp))[0]
        cands = (glob.glob(os.path.join(audio_dir, stem+'*mic*.wav')) or
                 glob.glob(os.path.join(audio_dir, stem+'*.wav')))
        if cands:
            records.append({'id': stem, 'audio': cands[0], 'jams': jp})
    return records

def guitarset_player(rec_id): return rec_id.split('_')[0]

def make_splits(records, held_out_players=('05',), val_frac=0.15, seed=0):
    import random; rng = random.Random(seed)
    test  = [r for r in records if guitarset_player(r['id']) in held_out_players]
    train_pool = [r for r in records if guitarset_player(r['id']) not in held_out_players]
    rng.shuffle(train_pool)
    n_val = int(len(train_pool) * val_frac)
    return {'train': train_pool[n_val:], 'val': train_pool[:n_val], 'test': test}

records = build_index(AUDIO_DIR, JAMS_DIR)
splits  = make_splits(records, held_out_players=('05',))
print(f"Total: {len(records)} | Train: {len(splits['train'])} | Val: {len(splits['val'])} | Test: {len(splits['test'])}")

# Sanity check
_n = load_guitarset_notes(records[0]['jams'])
print(f"Sample: {records[0]['id']} -> {len(_n)} notes | midi range {min(n['midi'] for n in _n)}-{max(n['midi'] for n in _n)}")

Total: 360 | Train: 255 | Val: 45 | Test: 60
Sample: 00_BN1-129-Eb_comp -> 133 notes | midi range 44-72


## 5. Segmentation + augmentation  ✅ unit-tested

In [5]:
import numpy as np, random, librosa

def segment_bounds(duration, seg=SEGMENT_SEC, hop=HOP_SEC):
    out=[]; t=0.0
    while t < duration:
        out.append((t, min(t+seg, duration)))
        if t+seg >= duration: break
        t += hop
    return out

def notes_in_segment(notes, t0, t1):
    seg=[]
    for n in notes:
        if n['onset'] < t1 and n['offset'] > t0:
            seg.append({'onset':  max(0.0, n['onset']-t0),
                        'offset': min(t1, n['offset'])-t0,
                        'midi':   n['midi']})
    return seg

def shift_notes(notes, semis):
    return [{**n, 'midi': n['midi']+semis} for n in notes]

def pitch_shift(y, sr, semis):
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=semis) if semis else y

# self-test
_gt = [{'onset':0.0,'offset':0.4,'midi':60},
       {'onset':0.5,'offset':0.9,'midi':64},
       {'onset':1.0,'offset':1.4,'midi':67}]
assert len(segment_bounds(12.0)) == 3
assert [n['midi'] for n in shift_notes(_gt,2)] == [62,66,69]
assert [n['midi'] for n in notes_in_segment(_gt,0.5,1.5)] == [64,67]
print("Segmentation + augmentation OK")

Segmentation + augmentation OK


## 6. Kong model targets — onset / offset / frame / velocity

In [6]:
# Kong's model predicts 4 targets (differs from Basic Pitch's 3):
# onset (T x 88), offset (T x 88), frame (T x 88), velocity (T x 88)
# Frame rate: Kong uses SR=16000, hop=160 -> 100 fps
# 10s segment -> 1000 frames

KONG_HOP    = 160          # samples
KONG_FPS    = SR / KONG_HOP  # 100 fps
N_FRAMES    = int(SEGMENT_SEC * KONG_FPS)   # 1000 frames per 10s segment
print(f"Kong frame rate: {KONG_FPS} fps | frames per segment: {N_FRAMES}")

def notes_to_kong_targets(notes, n_frames=N_FRAMES, fps=KONG_FPS,
                           midi_min=MIDI_MIN, n_notes=N_NOTES):
    onset    = np.zeros((n_frames, n_notes), np.float32)
    offset   = np.zeros((n_frames, n_notes), np.float32)
    frame    = np.zeros((n_frames, n_notes), np.float32)
    velocity = np.zeros((n_frames, n_notes), np.float32)
    for nt in notes:
        b = nt['midi'] - midi_min
        if not (0 <= b < n_notes): continue
        f0 = int(round(nt['onset']  * fps))
        f1 = max(f0+1, int(round(nt['offset'] * fps)))
        f0c, f1c = max(0,f0), min(n_frames,f1)
        frame[f0c:f1c, b]  = 1.0
        if 0 <= f0 < n_frames:   onset[f0, b]  = 1.0
        if 0 <= f1-1 < n_frames: offset[f1-1, b] = 1.0
        velocity[f0c:f1c, b] = 0.5   # neutral velocity (no GT velocity in GuitarSet)
    return onset, offset, frame, velocity

# self-test
_o,_off,_fr,_v = notes_to_kong_targets([{'onset':0.1,'offset':0.6,'midi':69}])
assert _fr[:, 69-MIDI_MIN].sum() > 0
print(f"Kong targets OK | shapes: {_o.shape} each")

Kong frame rate: 100.0 fps | frames per segment: 1000
Kong targets OK | shapes: (1000, 88) each


## 7. PyTorch Dataset

In [7]:
import torch
from torch.utils.data import Dataset, DataLoader

class GuitarDataset(Dataset):
    def __init__(self, records, augment=True):
        self.records = records
        self.augment = augment
        self._notes_cache = {}
        # Pre-build item list: (record, t0, t1)
        self.items = []
        import soundfile as sf
        for r in records:
            dur = sf.info(r['audio']).duration
            for (t0,t1) in segment_bounds(dur):
                if t1-t0 >= 2.0:
                    self.items.append((r, t0, t1))
        print(f"  Dataset: {len(records)} recordings -> {len(self.items)} segments")

    def _notes(self, r):
        if r['id'] not in self._notes_cache:
            self._notes_cache[r['id']] = load_guitarset_notes(r['jams'])
        return self._notes_cache[r['id']]

    def __len__(self): return len(self.items)

    def __getitem__(self, i):
        r, t0, t1 = self.items[i]
        y, _ = librosa.load(r['audio'], sr=SR, offset=t0,
                            duration=SEGMENT_SEC, mono=True)
        if len(y) < SEGMENT_SAMPLES:
            y = np.pad(y, (0, SEGMENT_SAMPLES - len(y)))
        y = y[:SEGMENT_SAMPLES]
        notes = notes_in_segment(self._notes(r), t0, t0+SEGMENT_SEC)
        if self.augment:
            semi = random.choice(PITCH_SHIFTS)
            if semi:
                y     = pitch_shift(y, SR, semi)
                notes = shift_notes(notes, semi)
        onset, offset, frame, velocity = notes_to_kong_targets(notes)
        return (torch.from_numpy(y).float(),
                torch.from_numpy(onset).float(),
                torch.from_numpy(offset).float(),
                torch.from_numpy(frame).float(),
                torch.from_numpy(velocity).float())

print("Building datasets...")
train_dataset = GuitarDataset(splits['train'], augment=True)
val_dataset   = GuitarDataset(splits['val'],   augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

Building datasets...
  Dataset: 255 recordings -> 5564 segments
  Dataset: 45 recordings -> 1036 segments
Train batches: 1391 | Val batches: 259


## 8. Load Kong model with pretrained weights

In [8]:
from piano_transcription_inference import PianoTranscription
from piano_transcription_inference.pytorch_utils import move_data_to_device
import torch.nn as nn

# Load Kong model (downloads pretrained weights automatically ~120MB)
print("Loading Kong model (downloads pretrained weights if needed)...")
transcriptor = PianoTranscription(device=device, checkpoint_path=None)
model = transcriptor.model
model = model.to(device)
model.train()

# Count parameters
total  = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total:,} | Trainable: {trainable:,}")

# Quick forward pass to confirm shapes
test_audio = torch.zeros(1, SEGMENT_SAMPLES).to(device)
with torch.no_grad():
    out = model(test_audio)
print("Output keys:", list(out.keys()))
for k,v in out.items(): print(f"  {k}: {v.shape}")

Loading Kong model (downloads pretrained weights if needed)...
Checkpoint path: /root/piano_transcription_inference_data/note_F1=0.9677_pedal_F1=0.9186.pth
Total size: ~165 MB
Using cuda for inference.
GPU number: 1
Total params: 42,946,305 | Trainable: 34,080,055
Output keys: ['reg_onset_output', 'reg_offset_output', 'frame_output', 'velocity_output', 'reg_pedal_onset_output', 'reg_pedal_offset_output', 'pedal_frame_output']
  reg_onset_output: torch.Size([1, 1001, 88])
  reg_offset_output: torch.Size([1, 1001, 88])
  frame_output: torch.Size([1, 1001, 88])
  velocity_output: torch.Size([1, 1001, 88])
  reg_pedal_onset_output: torch.Size([1, 1001, 1])
  reg_pedal_offset_output: torch.Size([1, 1001, 1])
  pedal_frame_output: torch.Size([1, 1001, 1])


## 9. Training loop — TART recipe

In [23]:
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
import os, numpy as np

def bce(pred, target):
    return nn.functional.binary_cross_entropy(torch.sigmoid(pred), target)

def train_step(batch, model, optimizer):
    audio, t_onset, t_offset, t_frame, t_vel = [x.to(device) for x in batch]
    model.train()
    optimizer.zero_grad()
    out = model(audio)
    # Trim model output to match target size (model outputs 1001, targets are 1000)
    T = t_onset.shape[1]
    loss = (bce(out['reg_onset_output'][:, :T, :],  t_onset)  +
            bce(out['reg_offset_output'][:, :T, :], t_offset) +
            bce(out['frame_output'][:, :T, :],      t_frame)  +
            bce(out['velocity_output'][:, :T, :],   t_vel))
    loss.backward()
    optimizer.step()
    return loss.item()

def val_loss_fn(val_loader, model, n_batches=20):
    model.eval(); losses=[]
    with torch.no_grad():
        for i, batch in enumerate(val_loader):
            if i >= n_batches: break
            audio, t_on, t_off, t_fr, t_vel = [x.to(device) for x in batch]
            out = model(audio)
            T = t_on.shape[1]
            loss = (bce(out['reg_onset_output'][:, :T, :],  t_on)  +
                    bce(out['reg_offset_output'][:, :T, :], t_off) +
                    bce(out['frame_output'][:, :T, :],      t_fr)  +
                    bce(out['velocity_output'][:, :T, :],   t_vel))
            losses.append(loss.item())
    return float(np.mean(losses))

def run_training(model, train_loader, val_loader,
                 total_steps=TOTAL_STEPS, val_every=VAL_EVERY, out_dir=OUTPUT_DIR):
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = StepLR(optimizer, step_size=DECAY_STEPS, gamma=LR_DECAY)
    os.makedirs(out_dir, exist_ok=True)
    best_val = float('inf'); step = 0
    train_iter = iter(train_loader)
    import time; t0 = time.time()

    while step < total_steps:
        try: batch = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            batch = next(train_iter)

        loss = train_step(batch, model, optimizer)
        scheduler.step(); step += 1

        if step % 100 == 0:
            elapsed = time.time()-t0
            sps = step/elapsed
            eta_h = (total_steps-step)/sps/3600
            print(f"step {step:>6} | loss {loss:.4f} | {sps:.1f} steps/s | ETA {eta_h:.1f}h")

        if step % val_every == 0:
            vl = val_loss_fn(val_loader, model)
            print(f"step {step:>6} | VAL LOSS {vl:.4f}")
            if vl < best_val:
                best_val = vl
                torch.save(model.state_dict(),
                           os.path.join(out_dir, 'best_model.pt'))
                print(f"  -> saved (best val loss {best_val:.4f})")
            model.train()

    print(f"\nTraining done. Best val loss: {best_val:.4f}")
    return model

print("Training functions ready.")

Training functions ready.


In [24]:
train_dataset = GuitarDataset(splits['train'], augment=True)
val_dataset   = GuitarDataset(splits['val'],   augment=False)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=2, pin_memory=True)
print(f"N_FRAMES={N_FRAMES} | SEGMENT_SAMPLES={SEGMENT_SAMPLES}")
print(f"Train: {len(train_dataset)} segments | Val: {len(val_dataset)} segments")

  Dataset: 255 recordings -> 5564 segments
  Dataset: 45 recordings -> 1036 segments
N_FRAMES=1000 | SEGMENT_SAMPLES=160000
Train: 5564 segments | Val: 1036 segments


In [25]:
torch.cuda.empty_cache()
import gc; gc.collect()

387

In [27]:
# Freeze all layers except the final output heads
for name, param in model.named_parameters():
    if any(head in name for head in ['onset', 'offset', 'frame', 'velocity']):
        param.requires_grad = True
    else:
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} params")

Trainable: 34,079,139 / 42,946,305 params


In [28]:
torch.cuda.empty_cache()
import gc; gc.collect()

BATCH_SIZE = 4
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

In [30]:
import time, torch

# Test pure GPU speed with fake data (no data loading)
fake_audio  = torch.randn(4, SEGMENT_SAMPLES).to(device)
fake_onset  = torch.zeros(4, N_FRAMES, 88).to(device)
fake_offset = torch.zeros(4, N_FRAMES, 88).to(device)
fake_frame  = torch.zeros(4, N_FRAMES, 88).to(device)
fake_vel    = torch.zeros(4, N_FRAMES, 88).to(device)

optimizer_test2 = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad], lr=LR)

t0 = time.time()
for _ in range(50):
    model.train()
    optimizer_test2.zero_grad()
    out = model(fake_audio)
    T = fake_onset.shape[1]
    loss = (bce(out['reg_onset_output'][:,:T,:], fake_onset) +
            bce(out['reg_offset_output'][:,:T,:], fake_offset) +
            bce(out['frame_output'][:,:T,:], fake_frame) +
            bce(out['velocity_output'][:,:T,:], fake_vel))
    loss.backward()
    optimizer_test2.step()

elapsed = time.time()-t0
print(f"Pure GPU: 50 steps in {elapsed:.1f}s = {50/elapsed:.1f} steps/s")
print(f"Projected: {100000/(50/elapsed)/3600:.1f} hours")

OutOfMemoryError: CUDA out of memory. Tried to allocate 112.00 MiB. GPU 0 has a total capacity of 22.03 GiB of which 113.12 MiB is free. Including non-PyTorch memory, this process has 21.92 GiB memory in use. Of the allocated memory 21.52 GiB is allocated by PyTorch, and 161.82 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [31]:
torch.cuda.empty_cache()
import gc; gc.collect()

fake_audio  = torch.randn(1, SEGMENT_SAMPLES).to(device)
fake_onset  = torch.zeros(1, N_FRAMES, 88).to(device)
fake_offset = torch.zeros(1, N_FRAMES, 88).to(device)
fake_frame  = torch.zeros(1, N_FRAMES, 88).to(device)
fake_vel    = torch.zeros(1, N_FRAMES, 88).to(device)

optimizer_test2 = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad], lr=LR)

t0 = time.time()
for _ in range(20):
    model.train()
    optimizer_test2.zero_grad()
    out = model(fake_audio)
    T = fake_onset.shape[1]
    loss = (bce(out['reg_onset_output'][:,:T,:], fake_onset) +
            bce(out['reg_offset_output'][:,:T,:], fake_offset) +
            bce(out['frame_output'][:,:T,:], fake_frame) +
            bce(out['velocity_output'][:,:T,:], fake_vel))
    loss.backward()
    optimizer_test2.step()

elapsed = time.time()-t0
print(f"Pure GPU batch=1: 20 steps in {elapsed:.1f}s = {20/elapsed:.1f} steps/s")
print(f"Projected 100k steps: {100000/(20/elapsed)/3600:.1f} hours")

Pure GPU batch=1: 20 steps in 13.9s = 1.4 steps/s
Projected 100k steps: 19.3 hours


In [32]:
torch.cuda.empty_cache()
import gc; gc.collect()

BATCH_SIZE = 1
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=1, shuffle=False,
                          num_workers=2, pin_memory=True)

# Unfreeze all params for maximum signal in limited steps
for p in model.parameters(): p.requires_grad = True

cfg_steps = 5_000
cfg_val_every = 500

finetuned = run_training(model, train_loader, val_loader,
                         total_steps=cfg_steps,
                         val_every=cfg_val_every,
                         out_dir=OUTPUT_DIR)

step    100 | loss 2.7761 | 1.4 steps/s | ETA 1.0h
step    200 | loss 2.7732 | 1.4 steps/s | ETA 0.9h
step    300 | loss 2.7730 | 1.4 steps/s | ETA 0.9h
step    400 | loss 2.7729 | 1.4 steps/s | ETA 0.9h
step    500 | loss 2.7728 | 1.4 steps/s | ETA 0.9h
step    500 | VAL LOSS 2.7727
  -> saved (best val loss 2.7727)
step    600 | loss 2.7728 | 1.4 steps/s | ETA 0.9h
step    700 | loss 2.7727 | 1.4 steps/s | ETA 0.9h
step    800 | loss 2.7727 | 1.4 steps/s | ETA 0.8h
step    900 | loss 2.7727 | 1.4 steps/s | ETA 0.8h
step   1000 | loss 2.7727 | 1.4 steps/s | ETA 0.8h
step   1000 | VAL LOSS 2.7726
  -> saved (best val loss 2.7726)
step   1100 | loss 2.7727 | 1.4 steps/s | ETA 0.8h
step   1200 | loss 2.7727 | 1.4 steps/s | ETA 0.8h
step   1300 | loss 2.7726 | 1.4 steps/s | ETA 0.7h
step   1400 | loss 2.7727 | 1.4 steps/s | ETA 0.7h
step   1500 | loss 2.7726 | 1.4 steps/s | ETA 0.7h
step   1500 | VAL LOSS 2.7726
  -> saved (best val loss 2.7726)
step   1600 | loss 2.7726 | 1.4 steps/s | E

KeyboardInterrupt: 

## 10. Smoke test — confirm speed before full run

In [29]:
# Run 50 steps to confirm speed and loss direction before committing to full run
import time
print("Smoke test: 50 steps...")
optimizer_test = optim.Adam(model.parameters(), lr=LR)
train_iter_test = iter(train_loader)
t0 = time.time()
for i in range(50):
    try: batch = next(train_iter_test)
    except StopIteration:
        train_iter_test = iter(train_loader)
        batch = next(train_iter_test)
    loss = train_step(batch, model, optimizer_test)
elapsed = time.time()-t0
sps = 50/elapsed
eta_h = (TOTAL_STEPS/sps)/3600
print(f"50 steps in {elapsed:.1f}s = {sps:.1f} steps/s")
print(f"Projected time for {TOTAL_STEPS:,} steps: {eta_h:.1f} hours")
print(f"Final loss: {loss:.4f}")
print("\nIf steps/s > 5 and loss looks stable, proceed to full training.")

Smoke test: 50 steps...
50 steps in 71.8s = 0.7 steps/s
Projected time for 100,000 steps: 39.9 hours
Final loss: 2.7887

If steps/s > 5 and loss looks stable, proceed to full training.


## 11. Full training run

In [ ]:
# Only run after smoke test confirms speed is acceptable (target: >5 steps/s)
model = run_training(model, train_loader, val_loader,
                     total_steps=TOTAL_STEPS,
                     val_every=VAL_EVERY,
                     out_dir=OUTPUT_DIR)

## 12. Evaluate — P50/R50/F50 on held-out test set

Run baseline (pretrained) first, then fine-tuned. The delta is your result.

In [ ]:
from piano_transcription_inference import PianoTranscription
import mir_eval, numpy as np, pandas as pd

ONSET_THRESHOLD = 0.40   # your tuned value
FRAME_THRESHOLD = 0.10   # Kong default

def decode_notes_kong(audio_path, model, sr=SR,
                      onset_thresh=ONSET_THRESHOLD,
                      frame_thresh=FRAME_THRESHOLD):
    """Run Kong model inference on a full audio file -> note list."""
    y, _ = librosa.load(audio_path, sr=sr, mono=True)
    model.eval()
    all_onset=[]; all_offset=[]; all_frame=[]; all_vel=[]
    for t0,t1 in segment_bounds(len(y)/sr):
        s0=int(t0*sr); chunk=y[s0:s0+SEGMENT_SAMPLES]
        if len(chunk)<SEGMENT_SAMPLES: chunk=np.pad(chunk,(0,SEGMENT_SAMPLES-len(chunk)))
        x = torch.from_numpy(chunk).float().unsqueeze(0).to(device)
        with torch.no_grad(): out=model(x)
        all_onset.append(torch.sigmoid(out['onset_output']).cpu().numpy()[0])
        all_offset.append(torch.sigmoid(out['offset_output']).cpu().numpy()[0])
        all_frame.append(torch.sigmoid(out['frame_output']).cpu().numpy()[0])
        all_vel.append(torch.sigmoid(out['velocity_output']).cpu().numpy()[0])
    onset_mat = np.concatenate(all_onset,  axis=0)
    frame_mat = np.concatenate(all_frame,  axis=0)
    # Simple decoding: find onset peaks above threshold, track frame activity
    notes=[]
    for b in range(N_NOTES):
        midi = b + MIDI_MIN
        in_note=False; note_start=0
        for f in range(len(frame_mat)):
            active = frame_mat[f,b] > frame_thresh
            is_onset = onset_mat[f,b] > onset_thresh
            if is_onset and not in_note:
                in_note=True; note_start=f
            elif in_note and not active:
                notes.append({'onset': note_start/KONG_FPS,
                              'offset': f/KONG_FPS, 'midi': midi})
                in_note=False
        if in_note:
            notes.append({'onset': note_start/KONG_FPS,
                          'offset': len(frame_mat)/KONG_FPS, 'midi': midi})
    return sorted(notes, key=lambda n: n['onset'])

def match_notes(gt, pred, onset_tol=ONSET_TOL):
    cands=[]
    for pi,p in enumerate(pred):
        for gi,g in enumerate(gt):
            if int(p['midi'])!=int(g['midi']): continue
            if abs(p['onset']-g['onset'])<=onset_tol: cands.append((abs(p['onset']-g['onset']),pi,gi))
    cands.sort(key=lambda x:x[0]); up,ug=set(),set()
    for dt,pi,gi in cands:
        if pi in up or gi in ug: continue
        up.add(pi); ug.add(gi)
    tp=len(up); fp=len(pred)-tp; fn=len(gt)-tp
    P=tp/(tp+fp) if tp+fp else 0; R=tp/(tp+fn) if tp+fn else 0
    F=2*P*R/(P+R) if P+R else 0
    return P,R,F

def evaluate_model(model, test_records, label="model"):
    rows=[]
    for i,rec in enumerate(test_records):
        gt   = load_guitarset_notes(rec['jams'])
        pred = decode_notes_kong(rec['audio'], model)
        P,R,F = match_notes(gt, pred)
        rows.append({'id':rec['id'],'P50':P,'R50':R,'F50':F,'n_gt':len(gt),'n_pred':len(pred)})
        if (i+1)%10==0: print(f"  {i+1}/{len(test_records)} done...")
    df=pd.DataFrame(rows)
    agg={'P50': float(np.average(df.P50,weights=df.n_gt)),
         'R50': float(np.average(df.R50,weights=df.n_gt)),
         'F50': float(np.average(df.F50,weights=df.n_gt))}
    print(f"\n── {label} ──")
    print(f"  P50: {agg['P50']:.4f} | R50: {agg['R50']:.4f} | F50: {agg['F50']:.4f}")
    print(f"  recordings: {len(df)} | avg gt notes: {df.n_gt.mean():.0f}")
    return df, agg

# ── Baseline: pretrained Kong (run before or after training — weights unchanged) ──
print("Evaluating PRETRAINED Kong model on held-out test set...")
base_model = PianoTranscription(device=device, checkpoint_path=None).model.to(device)
base_df, base_agg = evaluate_model(base_model, splits['test'], label="Pretrained Kong")

In [ ]:
# ── Fine-tuned: load best checkpoint ──
print("Evaluating FINE-TUNED model...")
ft_model = PianoTranscription(device=device, checkpoint_path=None).model.to(device)
ft_model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, 'best_model.pt'), map_location=device))
ft_df, ft_agg = evaluate_model(ft_model, splits['test'], label="Fine-tuned Kong (Acoustic)")

print(f"\n{'':25s} {'P50':>7} {'R50':>7} {'F50':>7}")
print(f"{'Pretrained Kong':25s} {base_agg['P50']:>7.4f} {base_agg['R50']:>7.4f} {base_agg['F50']:>7.4f}")
print(f"{'Fine-tuned (Acoustic)':25s} {ft_agg['P50']:>7.4f} {ft_agg['R50']:>7.4f} {ft_agg['F50']:>7.4f}")
print(f"{'Delta':25s} {ft_agg['P50']-base_agg['P50']:>+7.4f} {ft_agg['R50']-base_agg['R50']:>+7.4f} {ft_agg['F50']-base_agg['F50']:>+7.4f}")
print(f"\nNote: Basic Pitch baseline on same test set = 0.7758 F50 (your existing result)")